# Lab 2B: Direct APIM Access (No Foundry Spoke)

Demonstrate how teams can access the Landing Zone models **directly via APIM** without a Foundry spoke.

## Use Case: Northwind Legacy Integration

Some teams may:
- Have existing applications that can't use the Agent Service
- Need simple REST API access to models
- Be migrating from other platforms
- Want lightweight integration without Foundry overhead

## Step 1: Load APIM Configuration

In [1]:
import os
from pathlib import Path
# Load the .env file
env_file = Path("../../.env")
with open(env_file) as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            key, value = line.split('=', 1)
            os.environ[key] = value

APIM_URL = os.environ['APIM_URL']
APIM_KEY = os.environ['APIM_KEY']

print(f"APIM URL: {APIM_URL}")
print(f"APIM Key: {APIM_KEY[:8]}... (hidden)")

APIM URL: https://foundry-apim-xodlwq.azure-api.net/openai
APIM Key: 7848f0b2... (hidden)


## Step 2: Create a Dedicated APIM Subscription for Northwind

For production, each team should have their own APIM subscription for tracking and access control.

In [3]:
import subprocess
import json

RG = "lab1a-foundry-lz-hub"
APIM_NAME = APIM_URL.split('//')[1].split('.')[0]  # Extract APIM name from URL
SUB_ID = subprocess.run('az account show --query id -o tsv', shell=True, capture_output=True, text=True).stdout.strip()

# Create a subscription for Northwind
NORTHWIND_SUB_NAME = "northwind-legacy-access"

sub_body = {
    "properties": {
        "displayName": "Northwind Legacy Integration Access",
        "scope": f"/apis/openai",
        "state": "active"
    }
}

# Create via REST API
uri = f"https://management.azure.com/subscriptions/{SUB_ID}/resourceGroups/{RG}/providers/Microsoft.ApiManagement/service/{APIM_NAME}/subscriptions/{NORTHWIND_SUB_NAME}?api-version=2024-06-01-preview"

import tempfile
with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
    json.dump(sub_body, f)
    body_file = f.name

result = subprocess.run(
    f'az rest --method PUT --uri "{uri}" --body @{body_file} -o json',
    shell=True, capture_output=True, text=True
)

os.unlink(body_file)

if result.returncode == 0:
    print(f"✅ Created APIM subscription: {NORTHWIND_SUB_NAME}")
else:
    print(f"ℹ️ Subscription may already exist: {result.stderr[:100]}")

✅ Created APIM subscription: northwind-legacy-access


In [4]:
# Get the subscription key
key_uri = f"https://management.azure.com/subscriptions/{SUB_ID}/resourceGroups/{RG}/providers/Microsoft.ApiManagement/service/{APIM_NAME}/subscriptions/{NORTHWIND_SUB_NAME}/listSecrets?api-version=2024-06-01-preview"

result = subprocess.run(
    f'az rest --method POST --uri "{key_uri}" --query primaryKey -o tsv',
    shell=True, capture_output=True, text=True
)

NORTHWIND_KEY = result.stdout.strip()
print(f"Northwind APIM Key: {NORTHWIND_KEY[:8]}... (hidden)")

Northwind APIM Key: 16f90512... (hidden)


## Step 3: Use AzureOpenAI SDK with APIM

Use `AzureOpenAI` client with `api_key` authentication pointing to APIM.

In [5]:
!pip install openai -q

In [6]:
from openai import AzureOpenAI

# AzureOpenAI with APIM - uses api_key auth (not Entra)
client = AzureOpenAI(
    azure_endpoint=APIM_URL.replace("/openai", ""),  # Base without /openai suffix
    api_key=NORTHWIND_KEY,
    api_version="2024-10-21"
)

print("✅ AzureOpenAI client configured for APIM")
print(f"Endpoint: {APIM_URL.replace('/openai', '')}")

✅ AzureOpenAI client configured for APIM
Endpoint: https://foundry-apim-xodlwq.azure-api.net


## Step 4: Test Direct Chat Completions

Call models directly without Agent Service!

In [7]:
# Test with gpt-4.1-mini
print("Testing gpt-4.1-mini via direct APIM...")

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "system", "content": "You are a space exploration expert for the Galactic Discovery Agency."},
        {"role": "user", "content": "What are the best planets to visit for a first-time space tourist?"}
    ],
    max_tokens=150
)

print(f"\n✅ Model: {response.model}")
print(f"Response: {response.choices[0].message.content}")

Testing gpt-4.1-mini via direct APIM...

✅ Model: gpt-4.1-mini-2025-04-14
Response: For a first-time space tourist, the best destinations are those that are relatively safe, accessible, and offer spectacular experiences without requiring extensive training or risk. Given current and near-future technologies, here are the top options:

1. **Low Earth Orbit (LEO)**
   - *Description:* Orbiting roughly 160 to 2,000 kilometers above Earth.
   - *Why visit:* Experience weightlessness, see the Earth’s curvature, and witness stunning sunrises/sunsets. Companies like SpaceX, Blue Origin, and Axiom Space currently offer or plan to offer these trips.
   - *Accessibility:* Most accessible and safest; no need for deep-space travel.

2. **The Moon**
   - *Description:* Earth's


In [8]:
# Test with different prompts
prompts = [
    "Name the closest star to Earth.",
    "How many moons does Jupiter have?",
    "What causes a solar eclipse?"
]

print("Testing gpt-4.1-mini with space-themed prompts:")
print("="*50)

for prompt in prompts:
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=30
    )
    print(f"Q: {prompt}")
    print(f"A: {response.choices[0].message.content}\n")

Testing gpt-4.1-mini with space-themed prompts:
Q: Name the closest star to Earth.
A: The closest star to Earth is the Sun.

Q: How many moons does Jupiter have?
A: As of 2024, Jupiter has 95 confirmed moons.

Q: What causes a solar eclipse?
A: A solar eclipse occurs when the Moon passes between the Earth and the Sun, blocking all or part of the Sun's light from reaching the Earth. This



## Step 5: Streaming Example

Direct APIM access also supports streaming!

In [9]:
print("Streaming response from gpt-4.1-mini:")
print("-" * 40)

stream = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "user", "content": "Write a haiku about exploring Mars."}],
    stream=True
)

for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)

print("\n" + "-" * 40)
print("✅ Streaming complete!")

Streaming response from gpt-4.1-mini:
----------------------------------------
Red dust swirls gently,  
Silent plains stretch endlessly —  
Dreams walk on new soil.
----------------------------------------
✅ Streaming complete!


## Step 6: Using with LangChain (Optional)

LangChain also works with direct APIM access!

In [10]:
!pip install langchain-openai -q

In [11]:
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    azure_endpoint=APIM_URL.replace("/openai", ""),
    api_key=NORTHWIND_KEY,
    api_version="2024-10-21",
    azure_deployment="gpt-4.1-mini"
)

response = llm.invoke("What is the Great Red Spot on Jupiter?")
print(f"LangChain via APIM: {response.content}")

PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


LangChain via APIM: The Great Red Spot on Jupiter is a massive, persistent storm located in the planet's southern hemisphere. It is essentially a gigantic, high-pressure anticyclonic storm, similar to a hurricane on Earth but far larger and longer-lasting. The Great Red Spot has been observed for at least 350 years, making it one of the longest-known atmospheric phenomena in the solar system.

Key characteristics of the Great Red Spot include:

- **Size:** It is enormous—large enough to fit about two to three Earths across.
- **Color:** The reddish hue is due to complex chemical compounds in Jupiter's atmosphere, possibly involving phosphorus, sulfur, or organic molecules influenced by sunlight.
- **Winds:** Winds within the storm can reach speeds of up to 432 km/h (268 mph).
- **Duration:** It has been continuously observed since the 17th century, showing remarkable longevity.

Overall, the Great Red Spot is an iconic feature of Jupiter's turbulent atmosphere and a subject of extensiv

## Summary: Direct APIM vs Agent Service

### Configuration Pattern

```python
from openai import AzureOpenAI

client = AzureOpenAI(
    azure_endpoint="https://your-apim.azure-api.net",  # No /openai suffix
    api_key="your-apim-subscription-key",
    api_version="2024-10-21"
)
```

### When to Use

| Direct APIM | Agent Service |
|-------------|---------------|
| ✅ Migrating existing apps | ✅ New AI applications |
| ✅ Simple chat completions | ✅ Agent memory/state |
| ✅ Third-party frameworks | ✅ Tools & function calling |
| ✅ Streaming support | ✅ Multi-agent workflows |

**Next**: Lab 2C - Multi-Model Agents

## Save Northwind Configuration

In [12]:
from pathlib import Path

# Append to .env
env_file = Path("../../.env")
with open(env_file, 'a') as f:
    f.write(f"\n# Northwind Direct APIM Access\n")
    f.write(f"NORTHWIND_APIM_KEY={NORTHWIND_KEY}\n")

print(f"✅ Northwind configuration saved to .env")

✅ Northwind configuration saved to .env
